---
## STEP 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sqlalchemy import create_engine
import urllib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported successfully.')

Libraries imported successfully.


---
## STEP 1 — Data Ingestion

In [ ]:
print('=' * 60)
print('STEP 1: DATA INGESTION')
print('=' * 60)

RAW_PATH = 'Credit_Risk_Dataset.xlsx'
df_raw = pd.read_excel(RAW_PATH)

print('Dataset loaded successfully.')
print(f'   Rows    : {df_raw.shape[0]:,}')
print(f'   Columns : {df_raw.shape[1]}')

STEP 1: DATA INGESTION
Dataset loaded successfully.
   Rows    : 32,581
   Columns : 37


In [ ]:
df_raw.head(3)

First 3 rows

In [ ]:
print('Missing values per column:')
missing = df_raw.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print(f'   Total missing values: {missing.sum()}')
else:
    print(missing_cols)

Missing values per column:
   Total missing values: 0


In [ ]:
vc    = df_raw['loan_status'].value_counts()
total = len(df_raw)
print('Target Variable Distribution:')
print(f'   Non-Default (0): {vc[0]:,}   {vc[0]/total*100:.2f}%')
print(f'   Default     (1): {vc[1]:,}   {vc[1]/total*100:.2f}%')

Target Variable Distribution:
   Non-Default (0): 25,473   78.18%
   Default     (1):  7,108   21.82%


---
## STEP 2 — Data Cleaning

In [ ]:
print('=' * 60)
print('STEP 2: DATA CLEANING')
print('=' * 60)

df = df_raw.copy()

STEP 2: DATA CLEANING


In [ ]:
# Fix 1: Outlier ages > 90
age_outliers = (df['person_age'] > 90).sum()
age_median   = df.loc[df['person_age'] <= 90, 'person_age'].median()
df.loc[df['person_age'] > 90, 'person_age'] = age_median

print('Fix 1 - Outlier Ages (person_age > 90):')
print(f'   Outlier rows found : {age_outliers}')
print(f'   Replacement value  : {age_median} (median of valid ages)')
print('   Age outliers resolved.')

Fix 1 - Outlier Ages (person_age > 90):
   Outlier rows found : 0
   Replacement value  : 26.0 (median of valid ages)
   Age outliers resolved.


In [ ]:
# Fix 2: Missing employment length
miss_emp_before = df['person_emp_length'].isnull().sum()
df['person_emp_length'] = df.groupby('employment_type')['person_emp_length'] \
                            .transform(lambda x: x.fillna(x.median()))
miss_emp_after = df['person_emp_length'].isnull().sum()

print('Fix 2 - Missing Employment Length (group-median by employment_type):')
print(f'   Missing before : {miss_emp_before}')
print(f'   Missing after  : {miss_emp_after}')
print('   Employment length imputed.')

Fix 2 - Missing Employment Length (group-median by employment_type):
   Missing before : 0
   Missing after  : 0
   Employment length imputed.


In [ ]:
# Fix 3: Missing interest rate
miss_int_before = df['loan_int_rate'].isnull().sum()
df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'] \
                        .transform(lambda x: x.fillna(x.median()))
miss_int_after = df['loan_int_rate'].isnull().sum()

print('Fix 3 - Missing Interest Rate (group-median by loan_grade):')
print(f'   Missing before : {miss_int_before}')
print(f'   Missing after  : {miss_int_after}')
print('   Interest rate imputed.')

Fix 3 - Missing Interest Rate (group-median by loan_grade):
   Missing before : 0
   Missing after  : 0
   Interest rate imputed.


In [ ]:
# Fix 4: Drop redundant column
if 'loan_percent_income' in df.columns:
    df.drop(columns=['loan_percent_income'], inplace=True)
    print('Fix 4 - Drop redundant column: loan_percent_income')
    print('   Reason: identical to loan_to_income_ratio')
    print('   Column removed.')

Fix 4 - Drop redundant column: loan_percent_income
   Reason: identical to loan_to_income_ratio
   Column removed.


In [ ]:
print('Cleaning complete.')
print(f'   Total nulls remaining: {df.isnull().sum().sum()}')

Cleaning complete.
   Total nulls remaining: 0


---
## STEP 3 — Feature Engineering

In [ ]:
print('=' * 60)
print('STEP 3: FEATURE ENGINEERING')
print('=' * 60)

STEP 3: FEATURE ENGINEERING


In [ ]:
# 3A — Binary flags
df['prev_default']  = (df['cb_person_default_on_file'] == 'Y').astype(int)
df['default_label'] = df['loan_status'].map({0: 'Non-Default', 1: 'Default'})

print('3A - Binary flags:')
print('   prev_default  : 1 if cb_person_default_on_file == Y')
print('   default_label : Default / Non-Default')

3A - Binary flags:
   prev_default  : 1 if cb_person_default_on_file == Y
   default_label : Default / Non-Default


In [ ]:
# 3B — Age & Income buckets
df['age_group'] = pd.cut(
    df['person_age'],
    bins=[19, 25, 35, 45, 60, 200],
    labels=['20-25', '26-35', '36-45', '46-60', '60+']
)
df['income_group'] = pd.cut(
    df['person_income'],
    bins=[0, 30000, 60000, 100000, 200000, 1e9],
    labels=['<30K', '30-60K', '60-100K', '100-200K', '>200K']
)

print('3B - Age and Income buckets:')
print('\nage_group:')
print(df['age_group'].value_counts().sort_index().to_string())
print('\nincome_group:')
print(df['income_group'].value_counts().sort_index().to_string())

3B - Age and Income buckets:

age_group:
20-25    15352
26-35    13769
36-45     2814
46-60      582
60+         64

income_group:
<30K        4516
30-60K     14210
60-100K     9648
100-200K    3760
>200K        447


In [ ]:
# 3C — LTI & DTI tiers
df['lti_tier'] = pd.cut(
    df['loan_to_income_ratio'],
    bins=[0, 0.2, 0.4, 0.6, 1.0, 100],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)
df['dti_tier'] = pd.cut(
    df['debt_to_income_ratio'],
    bins=[0, 0.2, 0.4, 0.6, 1.0, 100],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

print('3C - LTI and DTI risk tiers:')
print('\nlti_tier:')
print(df['lti_tier'].value_counts().sort_index().to_string())
print('\ndti_tier:')
print(df['dti_tier'].value_counts().sort_index().to_string())

3C - LTI and DTI risk tiers:

lti_tier:
Very Low    22207
Low          9183
Medium       1139
High           52
Very High       0

dti_tier:
Very Low     4109
Low         18487
Medium       8675
High         1306
Very High       4


In [ ]:
# 3D — Composite risk score & tier
df['risk_score'] = (
    df['loan_to_income_ratio'].rank(pct=True)     * 0.25 +
    df['debt_to_income_ratio'].rank(pct=True)     * 0.20 +
    df['credit_utilization_ratio'].rank(pct=True) * 0.15 +
    df['past_delinquencies'].rank(pct=True)       * 0.20 +
    df['prev_default'].rank(pct=True)             * 0.20
) * 100

df['risk_tier'] = pd.cut(
    df['risk_score'],
    bins=[0, 25, 50, 75, 100],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Very High Risk']
)

print('3D - Composite risk score and tier:')
print('\nWeight breakdown:')
print('   LTI ratio          : 25%')
print('   DTI ratio          : 20%')
print('   Credit utilization : 15%')
print('   Past delinquencies : 20%')
print('   Prior default flag : 20%')
print('\nrisk_tier distribution:')
for tier, cnt in df['risk_tier'].value_counts().sort_index().items():
    print(f'   {tier:<18} {cnt:>6,}   {cnt/len(df)*100:.2f}%')

3D - Composite risk score and tier:

Weight breakdown:
   LTI ratio          : 25%
   DTI ratio          : 20%
   Credit utilization : 15%
   Past delinquencies : 20%
   Prior default flag : 20%

risk_tier distribution:
Low Risk           1073    3.29%
Medium Risk       15565   47.77%
High Risk         14529   44.59%
Very High Risk     1414    4.34%


In [ ]:
# 3E — Sort-order columns (astype str to avoid Categorical KeyError in pandas 2.0+)
df['loan_grade_order']   = df['loan_grade'].astype(str).map({'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7})
df['lti_tier_order']     = df['lti_tier'].astype(str).map({'Very Low':1,'Low':2,'Medium':3,'High':4,'Very High':5})
df['dti_tier_order']     = df['dti_tier'].astype(str).map({'Very Low':1,'Low':2,'Medium':3,'High':4,'Very High':5})
df['risk_tier_order']    = df['risk_tier'].astype(str).map({'Low Risk':1,'Medium Risk':2,'High Risk':3,'Very High Risk':4})
df['age_group_order']    = df['age_group'].astype(str).map({'20-25':1,'26-35':2,'36-45':3,'46-60':4,'60+':5})
df['income_group_order'] = df['income_group'].astype(str).map({'<30K':1,'30-60K':2,'60-100K':3,'100-200K':4,'>200K':5})

print('3E - Sort-order columns added for Power BI:')
print('   loan_grade_order   : A=1, B=2, C=3, D=4, E=5, F=6, G=7')
print('   lti_tier_order     : Very Low=1, Low=2, Medium=3, High=4, Very High=5')
print('   dti_tier_order     : Very Low=1, Low=2, Medium=3, High=4, Very High=5')
print('   risk_tier_order    : Low Risk=1, Medium Risk=2, High Risk=3, Very High Risk=4')
print('   age_group_order    : 20-25=1, 26-35=2, 36-45=3, 46-60=4, 60+=5')
print('   income_group_order : <30K=1, 30-60K=2, 60-100K=3, 100-200K=4, >200K=5')

3E - Sort-order columns added for Power BI:
   loan_grade_order   : A=1, B=2, C=3, D=4, E=5, F=6, G=7
   lti_tier_order     : Very Low=1, Low=2, Medium=3, High=4, Very High=5
   dti_tier_order     : Very Low=1, Low=2, Medium=3, High=4, Very High=5
   risk_tier_order    : Low Risk=1, Medium Risk=2, High Risk=3, Very High Risk=4
   age_group_order    : 20-25=1, 26-35=2, 36-45=3, 46-60=4, 60+=5
   income_group_order : <30K=1, 30-60K=2, 60-100K=3, 100-200K=4, >200K=5


In [ ]:
# 3F — Early Warning Flag
df['early_warning_flag'] = (
    (df['debt_to_income_ratio'] > 0.5) &
    (df['past_delinquencies']   >= 2)  &
    (df['cb_person_default_on_file'] == 'Y') &
    (df['loan_grade'].astype(str).isin(['D', 'E', 'F', 'G']))
).astype(int)

flagged    = df['early_warning_flag'].sum()
dr_flagged = df[df['early_warning_flag'] == 1]['loan_status'].mean() * 100
dr_others  = df[df['early_warning_flag'] == 0]['loan_status'].mean() * 100

print('3F - Early Warning Flag:')
print('   Conditions:')
print('     debt_to_income_ratio > 0.50')
print('     past_delinquencies   >= 2')
print('     cb_person_default_on_file = Y')
print('     loan_grade in D, E, F, G')
print(f'\n   Flagged borrowers      : {flagged}   ({flagged/len(df)*100:.2f}% of portfolio)')
print(f'   Default rate (flagged) : {dr_flagged:.2f}%')
print(f'   Default rate (others)  : {dr_others:.2f}%')

3F - Early Warning Flag:
   Conditions:
     debt_to_income_ratio > 0.50
     past_delinquencies   >= 2
     cb_person_default_on_file = Y
     loan_grade in D, E, F, G

   Flagged borrowers      : 37   (0.11% of portfolio)
   Default rate (flagged) : 75.68%
   Default rate (others)  : 21.76%


In [ ]:
print('=' * 60)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 60)
print(f'   Original columns : {df_raw.shape[1]}')
print(f'   Final columns    : {df.shape[1]}')
print(f'   Final rows       : {df.shape[0]:,}')
print(f'   Total nulls      : {df.isnull().sum().sum()}')
print('=' * 60)

FEATURE ENGINEERING COMPLETE
   Original columns : 37
   Final columns    : 43
   Final rows       : 32,581
   Total nulls      : 0


---
## STEP 3.5 — EDA Snapshot

In [ ]:
print('Default Rate by Loan Grade:')
grade_df = df.groupby('loan_grade').agg(
    Total_Loans  = ('loan_status', 'count'),
    Defaults     = ('loan_status', 'sum'),
    Default_Rate = ('loan_status', lambda x: round(x.mean() * 100, 2)),
    Avg_Int_Rate = ('loan_int_rate', lambda x: round(x.mean(), 2)),
    Avg_Loan_Amt = ('loan_amnt', lambda x: round(x.mean(), 2))
)
grade_df

Default Rate by Loan Grade:


In [ ]:
print('Default Rate by Home Ownership:')
df.groupby('person_home_ownership').agg(
    Count        = ('loan_status', 'count'),
    Defaults     = ('loan_status', 'sum'),
    Default_Rate = ('loan_status', lambda x: round(x.mean() * 100, 2))
).sort_values('Default_Rate', ascending=False)

Default Rate by Home Ownership:


In [ ]:
print('Default Rate by Income Group:')
df.groupby('income_group', observed=True).agg(
    Count        = ('loan_status', 'count'),
    Default_Rate = ('loan_status', lambda x: round(x.mean() * 100, 2))
)

Default Rate by Income Group:


In [ ]:
print('Portfolio Summary:')
print(f"   Total Loans       : {len(df):,}")
print(f"   Total Defaults    : {df['loan_status'].sum():,}   {df['loan_status'].mean()*100:.2f}%")
print(f"   Total Exposure    : ${df['loan_amnt'].sum():,}")
print(f"   Avg Loan Amount   : ${df['loan_amnt'].mean():,.0f}")
print(f"   Avg Interest Rate : {df['loan_int_rate'].mean():.2f}%")
print(f"   Avg Income        : ${df['person_income'].mean():,.0f}")
print(f"   Avg LTI Ratio     : {df['loan_to_income_ratio'].mean():.3f}")
print(f"   Avg DTI Ratio     : {df['debt_to_income_ratio'].mean():.3f}")

Portfolio Summary:
   Total Loans       : 32,581
   Total Defaults    :  7,108   21.82%
   Total Exposure    : $312,431,300
   Avg Loan Amount   : $9,589
   Avg Interest Rate : 11.01%
   Avg Income        : $66,075
   Avg LTI Ratio     : 0.171
   Avg DTI Ratio     : 0.345


---
## STEP 4 — Export to SQL Server (SSMS)

In [ ]:
print('=' * 60)
print('STEP 4: EXPORT TO SQL SERVER (SSMS)')
print('=' * 60)

SERVER_NAME   = r'DESKTOP-59NIIJT\SQLEXPRESS'
DATABASE_NAME = 'NovaBank_Risk'
TABLE_NAME    = 'loans'

params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER_NAME};'
    f'DATABASE={DATABASE_NAME};'
    f'Trusted_Connection=yes;'
)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

print('Connecting to SQL Server...')
print(f"Exporting table '{TABLE_NAME}' to {DATABASE_NAME}...")

df.to_sql(TABLE_NAME, engine, if_exists='replace', index=False)

print('Export successful.')
print(f'   Server   : {SERVER_NAME}')
print(f'   Database : {DATABASE_NAME}')
print(f'   Table    : {TABLE_NAME}')
print(f'   Rows     : {len(df):,}')
print(f'   Columns  : {df.shape[1]}')
print('=' * 60)
print('PROJECT PIPELINE FINISHED SUCCESSFULLY.')
print('=' * 60)

STEP 4: EXPORT TO SQL SERVER (SSMS)
Connecting to SQL Server...
Exporting table 'loans' to NovaBank_Risk...
Export successful.
   Server   : DESKTOP-59NIIJT\SQLEXPRESS
   Database : NovaBank_Risk
   Table    : loans
   Rows     : 32,581
   Columns  : 43
PROJECT PIPELINE FINISHED SUCCESSFULLY.
